# GCC1734 — IA | Companion 02
## Ajuste supervisionado (SFT): do texto às instruções

**Objetivo:** reutilizar o **mesmo modelo causal e o mesmo tokenizador** do Companion 01 e treiná-lo para produzir respostas demonstradas, sem confundir o SFT com o pré-treinamento.

Caminho didático: **checkpoint do pré-treinamento → dataset de instruções → tokens e máscara da perda → SFT → avaliação antes/depois**.

> **Pré-requisito:** execute primeiro o Companion 01 até a etapa que salva `artefatos_pretraining/modelo/` e `artefatos_pretraining/tokenizador/`. Os checkpoints **não** estão dentro deste pacote, pois precisam ser gerados durante o pré-treinamento. Não carregamos pesos da internet.

> **Limitação:** o modelo tem apenas duas camadas e foi pré-treinado com um corpus sintético reduzido. As tarefas e o formato foram deliberadamente simplificados. Melhora na perda de SFT não significa competência geral de seguimento de instruções.

## 1. Carregar checkpoint e examinar os dados

O novo dataset contém **960 pares** derivados de **320 frases do corpus de treinamento do Companion 01**. Cada frase produz três instruções, cujo alvo é calculado automaticamente:

- **Tema:** identificar o domínio (`animais`, `educacao`, `tecnologia`, `clima`);
- **Início:** retornar as duas primeiras palavras;
- **Final:** retornar a última palavra, sem pontuação.

O conteúdo não introduz um novo universo factual. O que muda é a supervisão **instrução → resposta**. Os pares de validação usam **frases distintas** das utilizadas no SFT, embora todas tenham aparecido no corpus original de pré-treinamento. Assim, a validação mede generalização do comportamento de SFT para novas instruções sobre conteúdos já vistos, e **não** fatos inéditos.

In [ ]:
from pathlib import Path
import inspect, math, random, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import transformers
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast, Trainer, TrainingArguments
from IPython.display import display

SEED = 1734
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE = Path(".")
DATA = BASE / "corpora" / "sft_pares.csv"
PT_MODEL = BASE / "artefatos_pretraining" / "modelo"
PT_TOKENIZER = BASE / "artefatos_pretraining" / "tokenizador"
assert DATA.exists(), f"Dataset ausente: {DATA.resolve()}"
assert (PT_MODEL / "config.json").exists(), (
    "Checkpoint ausente. Execute o Companion 01 até a célula de salvar o modelo e "
    "mantenha as pastas artefatos_pretraining/ junto deste notebook."
)
assert (PT_TOKENIZER / "tokenizer_config.json").exists(), "Tokenizador do Companion 01 ausente."

print("Torch:", torch.__version__, "| Transformers:", transformers.__version__)
print("Dispositivo:", "CUDA" if torch.cuda.is_available() else "CPU")

df = pd.read_csv(DATA)
print("Pares:", len(df))
display(pd.crosstab(df.tarefa, df.split, margins=True))
display(df[["instrucao", "resposta", "split"]].sample(9, random_state=SEED))
train_textos = set(df.loc[df.split == "treino", "texto"])
val_textos = set(df.loc[df.split == "validacao", "texto"])
assert train_textos.isdisjoint(val_textos), "Uma frase aparece em ambos os splits SFT."
assert len(df) == 960 and len(train_textos) == 256 and len(val_textos) == 64

In [ ]:
tokenizer = PreTrainedTokenizerFast.from_pretrained(PT_TOKENIZER)
model = GPT2LMHeadModel.from_pretrained(PT_MODEL)
assert model.config.vocab_size == len(tokenizer)
assert model.config.n_positions == 64
print("Vocabulário:", len(tokenizer), "| Parâmetros:", f"{model.num_parameters():,}")
print("Tokens especiais:", tokenizer.special_tokens_map)

# Observação importante: o tokenizador BPE NÃO é treinado novamente no SFT.
# Isso preserva os IDs e a matriz de embeddings do modelo pré-treinado.

## 2. Como o SFT representa instruções e respostas?

Utilizaremos o formato **`<bos>instrução
Resposta: resposta<eos>`**. Para gerar uma resposta, o modelo recebe **apenas o prefixo até `Resposta:`** e prevê os tokens seguintes.

Não adicionamos novos tokens especiais: utilizamos os tokens BOS/EOS existentes e a tokenização BPE aprendida no Companion 01. Isso preserva compatibilidade com a matriz de embeddings.

In [ ]:
def prefixo(instrucao):
    return instrucao + "\nResposta:"

def preparar_exemplo(instrucao, resposta):
    # Tokenização separada evita contar os tokens da instrução na perda.
    contexto = [tokenizer.bos_token_id] + tokenizer(prefixo(instrucao), add_special_tokens=False)["input_ids"]
    alvo = tokenizer(" " + resposta, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    ids = contexto + alvo
    labels = [-100] * len(contexto) + alvo
    return {"input_ids": ids, "labels": labels, "attention_mask": [1] * len(ids)}

r = df.iloc[0]
e = preparar_exemplo(r.instrucao, r.resposta)
print("Instrução:", r.instrucao)
print("Resposta demonstrada:", repr(r.resposta))
print("Texto decodificado:", tokenizer.decode(e["input_ids"]))
print("Tokens de contexto:", len(e["input_ids"]) - sum(v != -100 for v in e["labels"]))
print("Tokens da resposta (incluindo EOS):", sum(v != -100 for v in e["labels"]))

visual = pd.DataFrame({
    "posição": list(range(len(e["input_ids"]))),
    "token": tokenizer.convert_ids_to_tokens(e["input_ids"]),
    "id": e["input_ids"],
    "label": e["labels"],
    "contribui para a perda?": [v != -100 for v in e["labels"]],
})
display(visual)
assert all(v == -100 for v in e["labels"][:len(e["input_ids"]) - sum(v != -100 for v in e["labels"])])

### Atenção: `-100` e deslocamento autoregressivo

Os tokens da instrução entram no contexto, mas **não são alvos** da função de perda. Em modelos causais do `transformers`, o cálculo interno da perda desloca `logits` e `labels` em uma posição. Assim, o último token do contexto é responsável por prever o **primeiro token da resposta**.

O valor `-100` é o índice ignorado pela entropia cruzada do PyTorch. Ele não faz parte do vocabulário.

In [ ]:
model.eval()
with torch.no_grad():
    entradas = torch.tensor([e["input_ids"]])
    rotulos = torch.tensor([e["labels"]])
    saida = model(input_ids=entradas, labels=rotulos)
    logits = saida.logits
    loss_manual = F.cross_entropy(
        logits[:, :-1, :].reshape(-1, logits.size(-1)),
        rotulos[:, 1:].reshape(-1), ignore_index=-100,
    )
print("Loss produzida pelo modelo:", float(saida.loss))
print("Loss calculada manualmente:", float(loss_manual))
assert torch.allclose(saida.loss, loss_manual, atol=1e-5)

## 3. Preparar conjuntos e lotes

A separação treino/validação impede utilizar a mesma **frase** em ambos os conjuntos de SFT. Isso evita que o modelo veja no SFT uma versão da frase e seja avaliado em outra tarefa sobre a mesma frase.

Como o modelo pré-treinado tem **64 posições**, verificamos explicitamente o comprimento de cada exemplo. Exemplos longos são filtrados, nunca truncados silenciosamente: truncá-los poderia remover os tokens da resposta.

O `collator` preenche com `pad_token_id` e define `labels=-100` em todas as posições artificiais de preenchimento.

In [ ]:
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

MAX_LEN = model.config.n_positions

class SFTDataset(Dataset):
    def __init__(self, tabela):
        self.examples = []
        self.excluidos = 0
        for row in tabela.itertuples(index=False):
            example = preparar_exemplo(row.instrucao, row.resposta)
            if len(example["input_ids"]) <= MAX_LEN:
                self.examples.append(example)
            else:
                self.excluidos += 1
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        return self.examples[idx]

train_ds = SFTDataset(df[df.split == "treino"])
val_ds = SFTDataset(df[df.split == "validacao"])
print("Treino (usados/excluídos):", len(train_ds), train_ds.excluidos)
print("Validação (usados/excluídos):", len(val_ds), val_ds.excluidos)
assert len(train_ds) > 0 and len(val_ds) > 0

# Para resultados reproduzíveis, nunca remova exemplos sem mostrar contagens por tarefa.
if train_ds.excluidos + val_ds.excluidos:
    print("Algumas sequências excederam 64 posições. Avalie a cobertura por tarefa.")

class SFTCollator:
    def __init__(self, pad_id):
        self.pad_id = pad_id
    def __call__(self, batch):
        ids = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
        labs = [torch.tensor(x["labels"], dtype=torch.long) for x in batch]
        return {
            "input_ids": pad_sequence(ids, batch_first=True, padding_value=self.pad_id),
            "labels": pad_sequence(labs, batch_first=True, padding_value=-100),
            "attention_mask": pad_sequence(
                [torch.ones_like(v) for v in ids], batch_first=True, padding_value=0
            ),
        }

collator = SFTCollator(tokenizer.pad_token_id)
lote = collator([train_ds[i] for i in range(3)])
print({k: tuple(v.shape) for k, v in lote.items()})
print("Tokens do lote usados como alvos:", int((lote["labels"] != -100).sum()))
print("Tokens ignorados:", int((lote["labels"] == -100).sum()))
assert torch.all(lote["labels"][lote["attention_mask"] == 0] == -100)

## 4. Linha de base: modelo antes do SFT

Vamos registrar respostas **antes de atualizar os parâmetros**, para compará-las depois. A geração é **gulosa** (`do_sample=False`), evitando variação devida ao sorteio. O modelo pode continuar a instrução ou produzir tokens pouco adequados: ele não foi treinado especificamente para essas tarefas.

A geração começa com o mesmo formato de prefixo utilizado no SFT. Não usamos o texto correto da resposta como entrada.

In [ ]:
PROMPTS = [
    "Tema: O gato brinca na varanda.",
    "Tema: O programa processa os exemplos.",
    "Início: A professora estuda programação na sala.",
    "Final: O cachorro descansa no quintal.",
]

def responder(model, instrucao, max_new_tokens=14):
    model.eval()
    contexto = [tokenizer.bos_token_id] + tokenizer(prefixo(instrucao), add_special_tokens=False)["input_ids"]
    if len(contexto) + max_new_tokens > model.config.n_positions:
        max_new_tokens = max(1, model.config.n_positions - len(contexto))
    input_ids = torch.tensor([contexto], device=next(model.parameters()).device)
    with torch.no_grad():
        saida = model.generate(
            input_ids=input_ids, max_new_tokens=max_new_tokens, do_sample=False,
            eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
        )
    novos = saida[0, len(contexto):].tolist()
    return tokenizer.decode(novos, skip_special_tokens=True).strip()

antes = {p: responder(model, p) for p in PROMPTS}
display(pd.DataFrame([{"instrução": p, "resposta antes": antes[p]} for p in PROMPTS]))

## 5. Ajuste supervisionado com `Trainer`

Os pesos do pré-treinamento são o ponto inicial. A perda é a **entropia cruzada dos tokens da resposta demonstrada**, com máscara sobre o contexto. A taxa de aprendizado é menor que a usada no Companion 01 (`5e-4`).

Aumente `MAX_STEPS` se a aprendizagem ainda for insuficiente; diminua em máquinas lentas. O resultado depende do checkpoint anterior e da duração do treinamento.

In [ ]:
MAX_STEPS = 180
LEARNING_RATE = 2e-4
BATCH = 16

kw = dict(
    output_dir="artefatos_sft/temporario",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    logging_first_step=True,
    logging_steps=10,
    eval_steps=30,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    dataloader_num_workers=0,
)
sig_args = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in sig_args:
    kw["eval_strategy"] = "steps"
else:
    kw["evaluation_strategy"] = "steps"
args = TrainingArguments(**kw)
trainer_kw = dict(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, data_collator=collator)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kw["processing_class"] = tokenizer
else:
    trainer_kw["tokenizer"] = tokenizer
trainer = Trainer(**trainer_kw)
result = trainer.train()
metrics = trainer.evaluate()
print("Loss de validação após SFT:", round(metrics["eval_loss"], 4))

## 6. Curva de treinamento e comparação de respostas

A perda de validação é calculada sobre **respostas demonstradas para frases não usadas no SFT**. Ela mede a probabilidade das respostas de referência, não a qualidade geral de um assistente. Compare também as respostas produzidas na geração.

In [ ]:
historico = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(figsize=(8, 4))
for campo, nome in [("loss", "Treinamento"), ("eval_loss", "Validação")]:
    if campo in historico.columns:
        parte = historico.dropna(subset=[campo])
        ax.plot(parte["step"], parte[campo], marker="o", label=nome)
ax.set(xlabel="Atualizações", ylabel="Entropia cruzada", title="SFT — evolução da perda")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

depois = {p: responder(model, p) for p in PROMPTS}
display(pd.DataFrame([
    {"instrução": p, "antes do SFT": antes[p], "depois do SFT": depois[p]}
    for p in PROMPTS
]))

## 7. Avaliação por tarefa e generalização

Como as respostas são curtas e a tarefa é controlada, podemos medir **acerto exato** nos exemplos de validação. Essa métrica é mais rigorosa que a perda média, mas ainda só vale para nosso universo sintético.

Para evitar que a métrica seja distorcida por exemplos filtrados por comprimento, avaliamos apenas os exemplos que cabem no limite de 64 posições e mostramos o número de observações.

In [ ]:
avaliacao = []
for row in df[df.split == "validacao"].itertuples(index=False):
    ex = preparar_exemplo(row.instrucao, row.resposta)
    if len(ex["input_ids"]) > MAX_LEN:
        continue
    pred = responder(model, row.instrucao, max_new_tokens=12)
    esperado = str(row.resposta).strip()
    avaliacao.append({
        "tarefa": row.tarefa, "instrução": row.instrucao,
        "esperado": esperado, "predito": pred,
        "acerto": pred == esperado,
    })
resultados = pd.DataFrame(avaliacao)
assert len(resultados) > 0
print("Exemplos avaliados:", len(resultados))
display(resultados.groupby("tarefa").agg(exemplos=("acerto", "size"),
                                         acuracia=("acerto", "mean")))
display(resultados.sample(min(12, len(resultados)), random_state=SEED))

## 8. Inspecionar a probabilidade de uma resposta

Duas respostas podem ser linguisticamente plausíveis, mas apenas uma corresponde ao formato solicitado. Podemos calcular a **perda condicional** de uma resposta usando a máscara: isso mostra que o treinamento avalia a resposta **dado o contexto da instrução**, sem pontuar os tokens da própria instrução.

In [ ]:
def perda_resposta(model, instrucao, resposta):
    ex = preparar_exemplo(instrucao, resposta)
    ids = torch.tensor([ex["input_ids"]], device=next(model.parameters()).device)
    labels = torch.tensor([ex["labels"]], device=ids.device)
    model.eval()
    with torch.no_grad():
        return float(model(input_ids=ids, labels=labels).loss)

instrucao = "Tema: O gato brinca na varanda."
respostas = ["animais", "clima", "tecnologia", "educacao"]
display(pd.DataFrame([{"resposta": r, "perda média": perda_resposta(model, instrucao, r)}
                      for r in respostas]).sort_values("perda média"))
print("Atenção: comparar médias por token não equivale a comparar probabilidades totais",
      "de sequências com diferentes quantidades de tokens.")

## 9. Salvar modelo para o Companion 03 — alinhamento por preferências

O checkpoint do SFT será o ponto de partida da política e, em uma abordagem clássica de RLHF, pode também fornecer a política de referência. O tokenizador permanece exatamente o mesmo.

In [ ]:
DESTINO = Path("artefatos_sft")
(DESTINO / "modelo").mkdir(parents=True, exist_ok=True)
(DESTINO / "tokenizador").mkdir(parents=True, exist_ok=True)
model.save_pretrained(DESTINO / "modelo", safe_serialization=True)
tokenizer.save_pretrained(DESTINO / "tokenizador")
print("Modelo SFT:", (DESTINO / "modelo").resolve())
print("Tokenizador:", (DESTINO / "tokenizador").resolve())

## 10. Atividades para os alunos

1. **Alvos:** retire a máscara `-100` das posições da instrução. O que o modelo passará a otimizar? Por que isso muda a interpretação do SFT?
2. **Transferência:** compare a perda inicial do modelo pré-treinado e de um modelo com pesos aleatórios sobre o mesmo conjunto de SFT. O primeiro sempre terá vantagem? Justifique.
3. **Generalização:** compare acertos em frases de treinamento e em frases reservadas. Uma resposta correta para uma frase inédita prova competência geral de linguagem?
4. **Escala:** altere `MAX_STEPS`. Em que momento a perda de validação deixa de melhorar?
5. **Formato:** substitua a tarefa `Tema:` por uma instrução equivalente, por exemplo `Classifique:`. O modelo segue a intenção ou depende do formato aprendido?
6. **Limitações:** por que a perda do SFT não informa explicitamente qual de duas respostas alternativas é preferível?

**Próximo passo:** no Companion 03, criar pares `(instrução, resposta preferida, resposta rejeitada)` para demonstrar como um novo sinal pode orientar o alinhamento.

### Referências da API

- [Hugging Face — Trainer](https://huggingface.co/docs/transformers/main_classes/trainer)
- [Hugging Face — Causal language modeling](https://huggingface.co/docs/transformers/tasks/language_modeling)
- [PyTorch — CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)